# T2.3 – Unit Mapping Upload to DBRepo

**Vienna Weather Wet-Month Prediction Experiment**

This notebook uploads unit-of-measurement mappings for every numeric column
in our DBRepo schema via the REST API.

Every numeric attribute is mapped to a QUDT ontology URI.
QUDT is used as a practical fallback to the SI Digital Framework because it
provides stable, widely used URIs for all units needed in this weather dataset.

**Owner:** Person C  
**Ontology:** QUDT (https://qudt.org/)  
**Mapping file:** `docs/unit_mapping.csv`

## 1. Imports and configuration

We connect to DBRepo using the same credentials and IDs from T2.1.

In [28]:
import requests

ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
USERNAME = "azra1558"
PASSWORD = "Katalizator1558!"

DATABASE_ID = "899bfcba-7fec-40c9-9076-3a3a9372c844"

TABLE_IDS = {
    "weather_measurement": "2212bed4-ef8f-4d95-bb65-20b2adb28abd",
    "time_dimension":      "9f4dc236-81cc-4bdb-93e7-a8b6ed5436f2",
    "station":             "e6779029-ce40-4a9a-ad17-147e183dc757"
}

print("Configuration loaded.")

Configuration loaded.


## 2. Define unit mappings

This is the full unit mapping for every numeric column in all three tables.
Mappings follow the QUDT ontology URIs documented in `docs/unit_mapping.csv`.

In [29]:
# Each entry: (table_name, column_name, unit_uri, unit_label)
unit_mappings = [
    # weather_measurement — identifiers
    ("weather_measurement", "measurement_id", "http://www.ontology-of-units-of-measure.org/resource/om-2/one", "unitless"),
    ("weather_measurement", "station_num",    "http://www.ontology-of-units-of-measure.org/resource/om-2/one", "unitless"),
    ("weather_measurement", "time_id",        "http://www.ontology-of-units-of-measure.org/resource/om-2/one", "unitless"),
    # weather_measurement — temperature
    ("weather_measurement", "t_mean_c",     "http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius", "degree Celsius"),
    ("weather_measurement", "t_max_c",      "http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius", "degree Celsius"),
    ("weather_measurement", "t_min_c",      "http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius", "degree Celsius"),
    ("weather_measurement", "mean_t_max_c", "http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius", "degree Celsius"),
    ("weather_measurement", "mean_t_min_c", "http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius", "degree Celsius"),
    # weather_measurement — pressure
    ("weather_measurement", "p_mean_hpa", "http://www.ontology-of-units-of-measure.org/resource/om-2/hectopascal", "hectopascal"),
    ("weather_measurement", "p_max_hpa",  "http://www.ontology-of-units-of-measure.org/resource/om-2/hectopascal", "hectopascal"),
    ("weather_measurement", "p_min_hpa",  "http://www.ontology-of-units-of-measure.org/resource/om-2/hectopascal", "hectopascal"),
    # weather_measurement — precipitation
    ("weather_measurement", "precp_sum_mm", "http://www.ontology-of-units-of-measure.org/resource/om-2/millimetre", "millimetre"),
    ("weather_measurement", "num_precp_01", "http://www.ontology-of-units-of-measure.org/resource/om-2/one", "unitless count"),
    # weather_measurement — humidity
    ("weather_measurement", "rel_hum_pct",     "http://www.ontology-of-units-of-measure.org/resource/om-2/percent", "percent"),
    ("weather_measurement", "rel_hum_max_pct", "http://www.ontology-of-units-of-measure.org/resource/om-2/percent", "percent"),
    ("weather_measurement", "rel_hum_min_pct", "http://www.ontology-of-units-of-measure.org/resource/om-2/percent", "percent"),
    # weather_measurement — wind
    ("weather_measurement", "wind_vel_ms",     "http://www.ontology-of-units-of-measure.org/resource/om-2/metrePerSecond-Time", "metre per second"),
    ("weather_measurement", "wind_vel_max_ms", "http://www.ontology-of-units-of-measure.org/resource/om-2/metrePerSecond-Time", "metre per second"),
    ("weather_measurement", "num_wind_vel60",  "http://www.ontology-of-units-of-measure.org/resource/om-2/one", "unitless count"),
    # weather_measurement — sunshine and counts
    ("weather_measurement", "sun_h",      "http://www.ontology-of-units-of-measure.org/resource/om-2/hour", "hour"),
    ("weather_measurement", "num_clear",  "http://www.ontology-of-units-of-measure.org/resource/om-2/one", "unitless count"),
    ("weather_measurement", "num_cloud",  "http://www.ontology-of-units-of-measure.org/resource/om-2/one", "unitless count"),
    ("weather_measurement", "num_frost",  "http://www.ontology-of-units-of-measure.org/resource/om-2/one", "unitless count"),
    ("weather_measurement", "num_ice",    "http://www.ontology-of-units-of-measure.org/resource/om-2/one", "unitless count"),
    ("weather_measurement", "num_summer", "http://www.ontology-of-units-of-measure.org/resource/om-2/one", "unitless count"),
    ("weather_measurement", "num_heat",   "http://www.ontology-of-units-of-measure.org/resource/om-2/one", "unitless count"),
    # time_dimension
    ("time_dimension", "time_id",   "http://www.ontology-of-units-of-measure.org/resource/om-2/one",  "unitless"),
    ("time_dimension", "ref_year",  "http://www.ontology-of-units-of-measure.org/resource/om-2/year", "year"),
    ("time_dimension", "ref_month", "http://www.ontology-of-units-of-measure.org/resource/om-2/month", "month"),
    # station
    ("station", "station_num",       "http://www.ontology-of-units-of-measure.org/resource/om-2/one",    "unitless"),
    ("station", "district_code",     "http://www.ontology-of-units-of-measure.org/resource/om-2/one",    "unitless"),
    ("station", "sub_district_code", "http://www.ontology-of-units-of-measure.org/resource/om-2/one",    "unitless"),
    ("station", "latitude_deg",      "http://www.ontology-of-units-of-measure.org/resource/om-2/degree", "degree"),
    ("station", "longitude_deg",     "http://www.ontology-of-units-of-measure.org/resource/om-2/degree", "degree"),
    ("station", "altitude_m",        "http://www.ontology-of-units-of-measure.org/resource/om-2/metre",  "metre"),
]

print(f"Total mappings defined: {len(unit_mappings)}")

Total mappings defined: 35


## 3. Upload mappings to DBRepo

For each column, we call the DBRepo REST API to set the unit URI in the column metadata.
We print OK or FAILED for each one so we can see exactly what happened.

In [30]:
success = 0
failed = 0

for (table_name, column_name, unit_uri, unit_label) in unit_mappings:
    table_id = TABLE_IDS[table_name]
    
    # Get full table object with columns
    table = client.get_table(DATABASE_ID, table_id)
    
    # Find the column by name
    col = next((c for c in table.columns if c.name == column_name), None)
    if col is None:
        print(f"FAILED (column not found): {table_name}.{column_name}")
        failed += 1
        continue
    
    # Update the unit
    try:
        client.update_table_column(
            database_id=DATABASE_ID,
            table_id=table_id,
            column_id=col.id,
            unit_uri=unit_uri
        )
        print(f"OK: {table_name}.{column_name} → {unit_label}")
        success += 1
    except Exception as e:
        print(f"FAILED ({e}): {table_name}.{column_name}")
        failed += 1

print(f"\nDone. {success} succeeded, {failed} failed.")

OK: weather_measurement.measurement_id → unitless
OK: weather_measurement.station_num → unitless
OK: weather_measurement.time_id → unitless
OK: weather_measurement.t_mean_c → degree Celsius
OK: weather_measurement.t_max_c → degree Celsius
OK: weather_measurement.t_min_c → degree Celsius
OK: weather_measurement.mean_t_max_c → degree Celsius
OK: weather_measurement.mean_t_min_c → degree Celsius
OK: weather_measurement.p_mean_hpa → hectopascal
OK: weather_measurement.p_max_hpa → hectopascal
OK: weather_measurement.p_min_hpa → hectopascal
OK: weather_measurement.precp_sum_mm → millimetre
OK: weather_measurement.num_precp_01 → unitless count
OK: weather_measurement.rel_hum_pct → percent
OK: weather_measurement.rel_hum_max_pct → percent
OK: weather_measurement.rel_hum_min_pct → percent
OK: weather_measurement.wind_vel_ms → metre per second
OK: weather_measurement.wind_vel_max_ms → metre per second
OK: weather_measurement.num_wind_vel60 → unitless count
OK: weather_measurement.sun_h → hour
OK

## 4. Summary

The unit mappings have been uploaded to DBRepo for all numeric columns
across the `weather_measurement`, `time_dimension`, and `station` tables.

The full mapping is also documented statically in `docs/unit_mapping.csv`.